# 02 — Pré-processamento
Cada decisão precisa de justificativa escrita. Decidir não criar features é
aceitável, desde que o motivo esteja explícito.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

# Nome da variável alvo definida na etapa anterior
TARGET = "mau_pagador"

pd.set_option("display.max_columns", None)

# Carregando o arquivo que unimos e salvamos no final do Notebook 01
df = pd.read_csv('dados_unidos.csv')

## 1. Dados faltantes
Identificação de valores nulos e tratamento adequado para evitar exclusão massiva de registros.

In [3]:
nulos = df.isna().sum()
pd.DataFrame({"nulos": nulos, "pct": (nulos / len(df) * 100).round(2)}).query("nulos > 0")

,nulos,pct
OCCUPATION_TYPE,11323,31.06


**Decisão:** Preencher e Limpar.
A única variável com dados faltantes expressivos nesta base é a `OCCUPATION_TYPE` (Profissão), atingindo quase 31% da base. Como não podemos excluir quase um terço dos nossos clientes, decidimos preencher os valores nulos com a categoria "Desconhecido". Aproveitamos esta etapa para remover a coluna `ID`, pois ela é apenas um identificador de sistema e não possui valor preditivo para o modelo de crédito.

In [6]:
# Preenchendo nulos da profissão
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Desconhecido')

# Removendo o ID (ignora o erro se a coluna já tiver sido removida)
df = df.drop('ID', axis=1, errors='ignore')

## 2. Definição da variável alvo
Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção.

**Justificativa:** A binarização já foi aplicada diretamente durante o cruzamento das bases no Notebook 01.
O limiar adotado considerou como "Mau Pagador" (Classe 1) os clientes com *status* 2, 3, 4 e 5 no birô de crédito (ou seja, atrasos superiores a 60 dias). Os demais formam a Classe 0 (Bons Pagadores). Este limiar não foi por convenção, mas focado no contexto do negócio: atrasos curtos (até 30 dias) muitas vezes são falhas operacionais do cliente (esquecimento da fatura), enquanto atrasos acima de 60 dias representam risco real de inadimplência severa e perda de capital para a instituição.

## 3. Normalização / padronização
Escolha e justificativa: StandardScaler, MinMaxScaler ou nenhum?

**Justificativa:** Optamos pelo **StandardScaler**, pois variáveis como `AMT_INCOME_TOTAL` (Renda) possuem valores na casa dos milhares, enquanto `CNT_CHILDREN` (Filhos) são de unidades simples.
**(Data Leakage):** Seguindo as boas práticas sugeridas no template, **não** aplicaremos a escala neste notebook. O `StandardScaler` será aplicado estritamente no Notebook 03, após o *split* dos dados, sendo ajustado (`fit`) apenas nos dados de treino. Isso garante que não haja vazamento de informação estatística do teste para o treinamento.

## 4. Feature engineering
Justificativa: Transformação de dados categóricos.

**Decisão:** Como os algoritmos de *Machine Learning* (incluindo o Random Forest) exigem entradas numéricas, a nossa *Feature Engineering* principal será converter as variáveis de texto (como Estado Civil e Nível de Escolaridade) em colunas binárias, utilizando a técnica de *One-Hot Encoding*. Adotamos o `drop_first=True` para evitar o problema de multicolinearidade (dummy trap).

In [7]:
# Selecionar todas as colunas que são do tipo 'object' (texto)
colunas_texto = df.select_dtypes(include=['object']).columns

# Aplicar o One-Hot Encoding
df_processado = pd.get_dummies(df, columns=colunas_texto, drop_first=True)

df_processado.head()

,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,mau_pagador,CODE_GENDER_M,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_Y,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Desconhecido,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,0,427500.0,-12005,-4542,1,1,0,0,2.0,0,True,True,True,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,0,427500.0,-12005,-4542,1,1,0,0,2.0,0,True,True,True,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,0,112500.0,-21474,-1134,1,0,0,0,2.0,0,True,True,True,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,0,270000.0,-19110,-3051,1,0,1,1,1.0,0,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,0,270000.0,-19110,-3051,1,0,1,1,1.0,0,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False


## 5. Salvar dataset tratado
Exportando a base matematicamente preparada para a separação de treino e teste.

In [8]:
# Salvando o dataset pronto para a modelagem
df_processado.to_csv("dados_processados.csv", index=False)
print("Arquivo 'dados_processados.csv' gerado com sucesso!")

Arquivo 'dados_processados.csv' gerado com sucesso!
